# Piloto — Benchmark Sintético 2D — moondream2

Evalúa `moondream2` sobre el `.zip` generado con `2D.py` (modo por lotes). Sube ese `.zip` al panel de archivos de Colab antes de correr la celda de configuración.

## 1. Configuración de la corrida

Se usa el **100% de las preguntas** que vengan dentro del `.zip` — el rango de dificultad (cantidad de figuras) y cuántas imágenes por nivel ya se decidieron al generar el dataset con `2D.py`, esta pipeline no vuelve a muestrear ni filtrar nada.

In [1]:
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
MODEL_REVISION = "main"

BENCHMARK_NAME = "2Dsintetico"

ZIP_PATH = "/content/benchmark_2d_sintetico_lotes.zip"  # sube este archivo (generado con 2D.py, modo por lotes) al panel de archivos de Colab antes de correr esta celda
EXTRACT_DIR = "/content/dataset_2d_sintetico"

RANDOM_SEED = 42
TIMEOUT_SEGUNDOS = 60

OUTPUT_TXT_PATH = f"/content/resultados_{BENCHMARK_NAME}_{MODEL_ID.split('/')[-1]}.txt"
OUTPUT_CSV_PATH = f"/content/detalle_{BENCHMARK_NAME}_{MODEL_ID.split('/')[-1]}.csv"

print(f"Modelo:      {MODEL_ID} (revision={MODEL_REVISION})")
print(f"Benchmark:   {BENCHMARK_NAME}")
print(f"Zip fuente:  {ZIP_PATH}")
print(f"Salida .txt: {OUTPUT_TXT_PATH}")


Modelo:      Qwen/Qwen2-VL-2B-Instruct (revision=main)
Benchmark:   2Dsintetico
Zip fuente:  /content/benchmark_2d_sintetico_lotes.zip
Salida .txt: /content/resultados_2Dsintetico_Qwen2-VL-2B-Instruct.txt


## 2. Extracción del dataset

In [2]:
import zipfile, os

os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, "r") as zipf:
    zipf.extractall(EXTRACT_DIR)

IMAGES_DIR = os.path.join(EXTRACT_DIR, "images")
DATASET_JSON_PATH = os.path.join(EXTRACT_DIR, "dataset.json")

n_imgs = len([f for f in os.listdir(IMAGES_DIR) if f.endswith(".png")]) if os.path.isdir(IMAGES_DIR) else 0
print(f"Extraído en: {EXTRACT_DIR}")
print(f"Imágenes encontradas: {n_imgs}")
print(f"dataset.json presente: {os.path.exists(DATASET_JSON_PATH)}")


Extraído en: /content/dataset_2d_sintetico
Imágenes encontradas: 100
dataset.json presente: True


## 3. Carga del modelo

Esta celda es la **única que cambia entre modelos** (junto con la de configuración arriba). El resto del notebook (carga de datos, loop de inferencia, métricas, guardado) es idéntico para los 4 VLM.

In [3]:
# IMPORTANTE: se fija transformers==4.46.2, misma version ya validada en
# piloto_capture_qwen2vl_2b.ipynb (evita el bug de transformers>=5.0 con
# trust_remote_code: https://github.com/huggingface/transformers/issues/43883,
# y qwen_vl_utils no viene preinstalado en Colab).
!pip install -q "transformers==4.46.2" "huggingface_hub>=0.23" "qwen-vl-utils>=0.0.8" accelerate pillow psutil
print("Dependencias instaladas.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 18.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
Dependencias instaladas.


In [4]:
import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from qwen_vl_utils import process_vision_info
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo detectado: {device}")
if device == "cpu":
    print("ADVERTENCIA: sin GPU. La inferencia sera muy lenta.")

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()

# Qwen2-VL usa AutoProcessor (combina tokenizer + image processor) y
# Qwen2VLForConditionalGeneration. No requiere trust_remote_code ni
# preprocesamiento manual -- process_vision_info() extrae los inputs de imagen
# del mensaje estructurado antes de pasarlos al processor.
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
print(f"Processor cargado: {type(processor).__name__}")

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
).eval()

if device == "cuda":
    model = model.cuda()

vram_after_load_mb = (torch.cuda.max_memory_allocated() / 1024**2) if device == "cuda" else 0
print(f"Modelo cargado. VRAM tras la carga: {vram_after_load_mb:.1f} MB")

if model.generation_config.pad_token_id is None:
    model.generation_config.pad_token_id = model.generation_config.eos_token_id

# -- Interfaz comun --------------------------------------------------------
GENERATION_CONFIG = dict(max_new_tokens=64, do_sample=False)

def predecir(imagen, pregunta):
    """Recibe PIL.Image y str, devuelve str (respuesta cruda del modelo)."""
    messages = [{"role": "user", "content": [
        {"type": "image", "image": imagen},
        {"type": "text",  "text": pregunta},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        output_ids = model.generate(**inputs, **GENERATION_CONFIG)
    generated = output_ids[0][input_len:]
    return processor.decode(generated, skip_special_tokens=True).strip()
# ---------------------------------------------------------------------------

print("Interfaz predecir() lista.")


Dispositivo detectado: cuda


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (7 outputs left out)

`Qwen2VLRotaryEmbedding` can now be fully parameterized by passing the model config through the `config` argument. All other arguments will be removed in v4.46


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

Modelo cargado. VRAM tras la carga: 4215.2 MB
Interfaz predecir() lista.


## 4. Construcción de la muestra

Se toma el **100% de `dataset.json`**, sin muestreo. `num_shapes` se extrae del nombre de archivo (`escena_XXXX_nYYY.png`) cuando el dataset se generó en Modo por Lotes — sirve para el desglose de dificultad progresiva más abajo.

In [5]:
import json, re

with open(DATASET_JSON_PATH, "r", encoding="utf-8") as f:
    qa_data = json.load(f)

def extraer_num_shapes(img_filename):
    """Extrae la cantidad real de figuras desde el nombre de archivo (escena_XXXX_nYYY.png).
    Devuelve None si el archivo no trae el sufijo _nYYY (dataset generado en Modo Fijo)."""
    m = re.search(r"_n(\d+)\.png$", img_filename)
    return int(m.group(1)) if m else None

muestra = []
for item in qa_data:
    muestra.append({
        "image_path": os.path.join(IMAGES_DIR, item["image"]),
        "image_file": item["image"],
        "pregunta": item["question"],  # prompt oficial del benchmark sintetico, completo, sin sufijo
        "ground_truth": int(item["answer"]),
        "tipo_pregunta": item["type"],
        "num_shapes": extraer_num_shapes(item["image"]),
    })

print(f"Total de preguntas a evaluar: {len(muestra)} (100% de dataset.json, sin muestreo)")
print(f"Tipos de pregunta presentes: {sorted(set(m['tipo_pregunta'] for m in muestra))}")
print(f"Ejemplo: {muestra[0]}")


Total de preguntas a evaluar: 400 (100% de dataset.json, sin muestreo)
Tipos de pregunta presentes: ['conteo_color', 'conteo_combinado', 'conteo_forma', 'conteo_general']
Ejemplo: {'image_path': '/content/dataset_2d_sintetico/images/escena_0000_n005.png', 'image_file': 'escena_0000_n005.png', 'pregunta': 'How many shapes are there in total in the image?', 'ground_truth': 5, 'tipo_pregunta': 'conteo_general', 'num_shapes': 5}


## 5. Inferencia

Mismo parser de respuesta (`extraer_entero`) usado en CAPTURe/TallyQA — dígito primero, número en palabras como respaldo, `None` solo si de verdad no hay nada interpretable.

In [6]:
import time, concurrent.futures

NUMEROS_EN_PALABRAS = {
    "zero": 0, "none": 0, "no": 0, "one": 1, "two": 2, "three": 3, "four": 4,
    "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
    "eleven": 11, "twelve": 12, "thirteen": 13, "fourteen": 14, "fifteen": 15,
    "sixteen": 16, "seventeen": 17, "eighteen": 18, "nineteen": 19, "twenty": 20,
}

def extraer_entero(texto):
    """Extrae el conteo de la respuesta en texto libre del modelo.
    Primero busca un digito; si no hay, busca un numero escrito en palabras.
    Devuelve None solo si de verdad no se encuentra nada interpretable (parse fail real)."""
    match = re.search(r"-?\d+", texto)
    if match:
        return int(match.group())
    texto_limpio = re.sub(r"[^\w\s]", " ", texto.lower())
    for palabra, valor in NUMEROS_EN_PALABRAS.items():
        if re.search(rf"\b{palabra}\b", texto_limpio):
            return valor
    return None


def query_con_timeout(imagen, pregunta, timeout_seg):
    """Corre model.query en un hilo aparte con limite de tiempo (ver notebook de TallyQA para el detalle
    de la limitacion conocida: el hilo sigue en segundo plano si se cumple el timeout)."""
    executor = concurrent.futures.ThreadPoolExecutor(max_workers=1)
    future = executor.submit(predecir, imagen, pregunta)
    try:
        resultado = future.result(timeout=timeout_seg)
        executor.shutdown(wait=False)
        return resultado, False
    except concurrent.futures.TimeoutError:
        executor.shutdown(wait=False)
        return None, True


resultados = []
fallos_carga = 0
n_timeouts = 0

print(f"Iniciando inferencia sobre {len(muestra)} preguntas (timeout={TIMEOUT_SEGUNDOS}s)...\n")

for i, item in enumerate(muestra):
    try:
        img = Image.open(item["image_path"]).convert("RGB")
    except Exception as e:
        fallos_carga += 1
        print(f"[{i+1}/{len(muestra)}] Error abriendo imagen ({item['image_path']}): {e}")
        continue

    pregunta_modelo = item["pregunta"]

    t0 = time.time()
    respuesta_cruda, timed_out = query_con_timeout(img, pregunta_modelo, TIMEOUT_SEGUNDOS)
    t1 = time.time()

    if timed_out:
        n_timeouts += 1
        pred = None
        parse_fail = False
    else:
        pred = extraer_entero(respuesta_cruda)
        parse_fail = pred is None

    resultados.append({
        "image_file": item["image_file"],
        "tipo_pregunta": item["tipo_pregunta"],
        "num_shapes": item["num_shapes"],
        "pregunta": item["pregunta"],
        "ground_truth": item["ground_truth"],
        "respuesta_cruda": respuesta_cruda,
        "prediccion": pred,
        "parse_fail": parse_fail,
        "timeout": timed_out,
        "error_abs": abs(pred - item["ground_truth"]) if (pred is not None) else None,
        "acierto_exacto": (pred == item["ground_truth"]) if (pred is not None) else False,
        "tiempo_seg": t1 - t0,
    })

    if timed_out:
        print(f"[{i+1}/{len(muestra)}] [{item['tipo_pregunta']}] TIMEOUT tras {t1-t0:.1f}s -- se registra y se sigue.")
    elif (i + 1) % 25 == 0 or i == 0:
        print(f"[{i+1}/{len(muestra)}] [{item['tipo_pregunta']:>16}] GT={item['ground_truth']:>4} | Pred={str(pred):>4} | "
              f"'{respuesta_cruda}' | {t1-t0:.2f}s")

print(f"\nInferencia terminada: {len(resultados)} preguntas procesadas, "
      f"{fallos_carga} fallos de carga de imagen, {n_timeouts} timeouts.")


Iniciando inferencia sobre 400 preguntas (timeout=60s)...



/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.001` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


[1/400] [  conteo_general] GT=   5 | Pred=   5 | 'There are five shapes in the image.' | 10.88s
[25/400] [  conteo_general] GT=  10 | Pred=  10 | 'There are 10 shapes in the image.' | 11.10s
[50/400] [    conteo_forma] GT=   4 | Pred=   3 | 'There are three squares in the image.' | 11.24s
[75/400] [    conteo_color] GT=   1 | Pred=   2 | 'There are two red shapes in the image.' | 11.18s
[100/400] [conteo_combinado] GT=   0 | Pred=   2 | 'There are two blue diamonds in the image.' | 11.17s
[125/400] [  conteo_general] GT=  35 | Pred=  30 | 'There are 30 shapes in the image.' | 11.18s
[150/400] [    conteo_forma] GT=  10 | Pred=  10 | 'There are 10 diamonds in the image.' | 11.31s
[175/400] [    conteo_color] GT=   9 | Pred=  10 | 'There are 10 green shapes in the image.' | 11.24s
[200/400] [conteo_combinado] GT=   3 | Pred=   2 | 'There are 2 orange diamonds in the image.' | 11.19s
[225/400] [  conteo_general] GT=  60 | Pred=  50 | 'There are 50 shapes in total in the image.' | 11.37s
[

## 6. Métricas y desglose

Dos ángulos de desglose: por **tipo de pregunta** (conteo_general/conteo_forma/conteo_color/conteo_combinado) y por **num_shapes** (curva de dificultad progresiva, cuando el dataset viene del Modo por Lotes).

In [7]:
import pandas as pd
import psutil

df = pd.DataFrame(resultados)

vram_peak_mb = (torch.cuda.max_memory_allocated() / 1024**2) if device == "cuda" else 0
ram_peak_mb = psutil.Process().memory_info().rss / 1024**2

n_total = len(df)
n_timeouts_tot = int(df["timeout"].sum())
n_parse_fail_tot = int(df["parse_fail"].sum())
validos = df[(~df["parse_fail"]) & (~df["timeout"])]
tiempos = df[~df["timeout"]]

print("=" * 60)
print(f"RESULTADOS — 2D Sintético — {MODEL_ID}")
print("=" * 60)
print(f"VRAM pico: {vram_peak_mb:.1f} MB | RAM pico: {ram_peak_mb:.1f} MB")
print(f"Preguntas evaluadas:      {n_total}")
print(f"Timeout rate:             {n_timeouts_tot/n_total:.1%}  ({n_timeouts_tot}/{n_total})")
print(f"Parse fail rate:          {n_parse_fail_tot/n_total:.1%}  ({n_parse_fail_tot}/{n_total})")
print(f"Exactitud global:         {validos['acierto_exacto'].mean():.1%}  (sobre {len(validos)} válidas)")
print(f"MAE global:               {validos['error_abs'].mean():.3f}")
print(f"Tiempo medio/pregunta:    {tiempos['tiempo_seg'].mean():.2f}s (± {tiempos['tiempo_seg'].std():.2f}s)")

print("\n--- Desglose por tipo de pregunta ---")
resumen_tipo = validos.groupby("tipo_pregunta").agg(
    n=("acierto_exacto", "size"), accuracy=("acierto_exacto", "mean"), mae=("error_abs", "mean"),
)
for tipo, fila in resumen_tipo.iterrows():
    print(f"{tipo:>18}: n={int(fila['n']):>4} | accuracy={fila['accuracy']:.1%} | MAE={fila['mae']:.3f}")

resumen_num_shapes = None
if validos["num_shapes"].notna().any():
    print("\n--- Desglose por num_shapes (curva de dificultad progresiva) ---")
    resumen_num_shapes = validos.groupby("num_shapes").agg(
        n=("acierto_exacto", "size"), accuracy=("acierto_exacto", "mean"), mae=("error_abs", "mean"),
    ).sort_index()
    for n_shapes, fila in resumen_num_shapes.iterrows():
        aviso = "  <- n bajo" if fila["n"] < 10 else ""
        print(f"num_shapes={int(n_shapes):>4}: n={int(fila['n']):>4} | accuracy={fila['accuracy']:.1%} | MAE={fila['mae']:.3f}{aviso}")


RESULTADOS — 2D Sintético — Qwen/Qwen2-VL-2B-Instruct
VRAM pico: 8673.8 MB | RAM pico: 2949.7 MB
Preguntas evaluadas:      400
Timeout rate:             0.0%  (0/400)
Parse fail rate:          0.0%  (0/400)
Exactitud global:         22.2%  (sobre 400 válidas)
MAE global:               4.343
Tiempo medio/pregunta:    11.22s (± 0.25s)

--- Desglose por tipo de pregunta ---
      conteo_color: n= 100 | accuracy=28.0% | MAE=1.640
  conteo_combinado: n= 100 | accuracy=23.0% | MAE=1.270
      conteo_forma: n= 100 | accuracy=17.0% | MAE=4.230
    conteo_general: n= 100 | accuracy=21.0% | MAE=10.230

--- Desglose por num_shapes (curva de dificultad progresiva) ---
num_shapes=   5: n=  20 | accuracy=55.0% | MAE=0.600
num_shapes=  10: n=  20 | accuracy=60.0% | MAE=0.450
num_shapes=  15: n=  20 | accuracy=30.0% | MAE=0.900
num_shapes=  20: n=  20 | accuracy=25.0% | MAE=1.550
num_shapes=  25: n=  20 | accuracy=20.0% | MAE=1.650
num_shapes=  30: n=  20 | accuracy=55.0% | MAE=0.700
num_shapes=  35: 

## 7. Guardado de resultados (`detalle_*.csv` / `resultados_*.txt`)

In [8]:
import datetime

with open(OUTPUT_TXT_PATH, "w", encoding="utf-8") as f:
    f.write(f"BENCHMARK: {BENCHMARK_NAME}\n")
    f.write(f"MODELO: {MODEL_ID} (revision={MODEL_REVISION})\n")
    f.write(f"FECHA: {datetime.datetime.now().isoformat()}\n")
    f.write(f"N_TOTAL: {n_total}\n")
    f.write(f"SEMILLA: {RANDOM_SEED}\n")
    f.write("-" * 50 + "\n")
    f.write(f"TIMEOUT_RATE: {n_timeouts_tot/n_total:.4f}\n")
    f.write(f"PARSE_FAIL_RATE: {n_parse_fail_tot/n_total:.4f}\n")
    f.write(f"ACCURACY_GLOBAL: {validos['acierto_exacto'].mean():.4f}\n")
    f.write(f"MAE_GLOBAL: {validos['error_abs'].mean():.4f}\n")
    f.write(f"TIEMPO_MEDIO_SEG: {tiempos['tiempo_seg'].mean():.4f}\n")
    f.write(f"VRAM_PICO_MB: {vram_peak_mb:.2f}\n")
    f.write(f"RAM_PICO_MB: {ram_peak_mb:.2f}\n")
    f.write("-" * 50 + "\n")

    f.write("DESGLOSE_POR_TIPO_PREGUNTA\n")
    for tipo, fila in resumen_tipo.iterrows():
        f.write(f"  {tipo}: n={int(fila['n'])} accuracy={fila['accuracy']:.4f} mae={fila['mae']:.4f}\n")
    f.write("-" * 50 + "\n")

    if resumen_num_shapes is not None:
        f.write("DESGLOSE_POR_NUM_SHAPES\n")
        for n_shapes, fila in resumen_num_shapes.iterrows():
            f.write(f"  num_shapes={int(n_shapes)}: n={int(fila['n'])} accuracy={fila['accuracy']:.4f} mae={fila['mae']:.4f}\n")
        f.write("-" * 50 + "\n")

print(f"Resultados guardados en: {OUTPUT_TXT_PATH}")

df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Detalle completo guardado como CSV en: {OUTPUT_CSV_PATH}")

try:
    from google.colab import files
    files.download(OUTPUT_TXT_PATH)
    files.download(OUTPUT_CSV_PATH)
except ImportError:
    print("No se detectó entorno Colab -- descarga los archivos manualmente desde el panel de archivos.")


Resultados guardados en: /content/resultados_2Dsintetico_Qwen2-VL-2B-Instruct.txt
Detalle completo guardado como CSV en: /content/detalle_2Dsintetico_Qwen2-VL-2B-Instruct.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>